In [8]:
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns 
import numpy as np
import requests

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor 
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

In [9]:
#Bikes CSV data prep

bikes_hour_df = pd.read_csv(r"C:\Users\Russell\Bike\hour.csv")
bikes_df = bikes_hour_df.copy()

bikes_df['dteday'] = pd.to_datetime(bikes_df['dteday'])
print(bikes_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   instant     17379 non-null  int64         
 1   dteday      17379 non-null  datetime64[us]
 2   season      17379 non-null  int64         
 3   yr          17379 non-null  int64         
 4   mnth        17379 non-null  int64         
 5   hr          17379 non-null  int64         
 6   holiday     17379 non-null  int64         
 7   weekday     17379 non-null  int64         
 8   workingday  17379 non-null  int64         
 9   weathersit  17379 non-null  int64         
 10  temp        17379 non-null  float64       
 11  atemp       17379 non-null  float64       
 12  hum         17379 non-null  float64       
 13  windspeed   17379 non-null  float64       
 14  casual      17379 non-null  int64         
 15  registered  17379 non-null  int64         
 16  cnt         17379 non-null  int64

In [12]:
#Pulling the weather off with Open-Meteo API

url = "https://archive-api.open-meteo.com/v1/archive?latitude=38.907&longitude=-77.037&start_date=2011-01-01&end_date=2013-01-01&hourly=temperature_2m&timezone=America%2FNew_York"
response = requests.get(url)

print(response.status_code)
print(response.headers['Content-Type'])

data = response.json()

print(type(data))
print(data.keys())
print(data['hourly'].keys())
print(len(data['hourly']['time']))

API_time = pd.to_datetime(data['hourly']['time'])
API_temp = data['hourly']['temperature_2m']

weather_df = pd.DataFrame(data={
    'time': API_time,
    'temp': API_temp
})

200
application/json; charset=utf-8
<class 'dict'>
dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])
dict_keys(['time', 'temperature_2m'])
17568


In [24]:
#combine the weather_API and bike data

bikes_df['time_key'] = (bikes_df['dteday'] + pd.to_timedelta(bikes_df['hr'], unit='h'))
merged_df = bikes_df.merge(weather_df,
                           how='left',
                           left_on='time_key',
                           right_on='time'
                          )

merged_df = merged_df.rename(columns={'temp_x': 'temp_scaled', 'temp_y': 'temp', 'time_key': 'datetime'})
merged_df['day_of_month'] = merged_df['dteday'].dt.day

print(merged_df.head(5))

   instant     dteday  season  yr  mnth  hr  holiday  weekday  workingday  \
0        1 2011-01-01       1   0     1   0        0        6           0   
1        2 2011-01-01       1   0     1   1        0        6           0   
2        3 2011-01-01       1   0     1   2        0        6           0   
3        4 2011-01-01       1   0     1   3        0        6           0   
4        5 2011-01-01       1   0     1   4        0        6           0   

   weathersit  ...   atemp   hum  windspeed  casual  registered  cnt  \
0           1  ...  0.2879  0.81        0.0       3          13   16   
1           1  ...  0.2727  0.80        0.0       8          32   40   
2           1  ...  0.2727  0.80        0.0       5          27   32   
3           1  ...  0.2879  0.75        0.0       3          10   13   
4           1  ...  0.2879  0.75        0.0       0           1    1   

             datetime                time temp  day_of_month  
0 2011-01-01 00:00:00 2011-01-01 00:00:00

In [27]:
#Data Prep

y = merged_df['cnt']
X = merged_df.drop(labels=['cnt', 'registered', 'casual', 'instant', 'dteday', 'time', 'datetime'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2, 
                                                    random_state=42) 


X_num = ['temp_scaled', 'atemp', 'hum', 'windspeed', 'temp']
X_cat = ['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit', 'day_of_month', 'hr']


preprocessor_num = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
     ("scaler", StandardScaler())
])

preprocessor_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder())
])

preprocessor = ColumnTransformer(transformers=[
    ("preprocessor_num", preprocessor_num, X_num),
    ("preprocessor_cat", preprocessor_cat, X_cat)
])

In [29]:
# Baseline LinearRegression

baseline_linreg = Pipeline([
    ('preprocessor', preprocessor),
    ('linreg', LinearRegression())
])

baseline_linreg_cv = np.mean(cross_val_score(baseline_linreg, X_train, y_train, cv=5)) 
print(baseline_linreg_cv)

0.6842036130556732


In [30]:
#Baseline DecisionTree

baseline_DT = Pipeline([
    ('preprocessor', preprocessor),
    ('DT', DecisionTreeRegressor())
])

baseline_DT_cv = np.mean(cross_val_score(baseline_DT, X_train, y_train))
print(baseline_DT_cv)

0.8306208490806182


In [38]:
#Baseline RandomForestRegressor

baseline_RF = Pipeline([
    ('preprocessor', preprocessor),
    ('RF', RandomForestRegressor()) 
])

baseline_RF_cv = np.mean(cross_val_score(baseline_RF, X_train, y_train, cv=5))
print(baseline_RF_cv)

0.9163421293830405


In [36]:
#Baseline GradientBoostingRegressor

baseline_GBR = Pipeline([
    ('preprocessor', preprocessor),
    ('GBR', GradientBoostingRegressor())
])

baseline_GBR_cv = np.mean(cross_val_score(baseline_GBR, X_train, y_train, cv=5))
print(baseline_GBR_cv)

0.7940561589006835


In [37]:
#Baseline XGBRegressor

baseline_XGB = Pipeline([
    ('preprocessor', preprocessor),
    ('XGB', XGBRegressor())
])

baseline_XGB_cv = np.mean(cross_val_score(baseline_XGB, X_train, y_train, cv=5))
print(baseline_XGB_cv)

0.9313411116600037


In [48]:
model_summary = {
    'LinearRegression': baseline_linreg_cv,
    'DecisionTree': baseline_DT_cv,
    'RandomForest': baseline_RF_cv,
    'GradientBoosting': [baseline_GBR_cv,
    'XGB': [baseline_XGB_cv]
}

models_name = model_summary.keys()
print(models_name)

print(pd.DataFrame(data=model_summary, columns=models_name))

dict_keys(['LinearRegression', 'DecisionTree', 'RandomForest', 'GradientBoosting', 'XGB'])
   LinearRegression  DecisionTree  RandomForest  GradientBoosting       XGB
0          0.684204      0.830621      0.916342          0.794056  0.931341
